# 07 - Cross-model comparison

Reads Parquet outputs produced by `scripts/dev/build_revision_parquets.py`.
Adding a new model is a one-line change to `MODELS_TO_COMPARE` below.

**Outputs:** CSV tables under `notebooks/outputs/revision/` and PDF figures
under `notebooks/outputs/revision/figures/`.

## 1. Parameters

In [1]:
MODELS_TO_COMPARE = [
    'gpt-5',
    'deepseek-r1',
    'claude-sonnet-4-5-thinking',  # comment out until at least 1 full run is ingested
]
ANCHOR_MODEL = 'gpt-5'           # statistical comparisons (Wilcoxon) anchor on this model
QUESTIONNAIRES = ['pcs_total', 'bpi_total_mean', 'tsk_total']
BOOTSTRAP_ITERS = 5000
RNG_SEED = 42


## 2. Setup

In [2]:
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import combinations

notebook_dir = Path.cwd()
project_root = notebook_dir.parent
sys.path.insert(0, str(project_root / "src"))
from pain_narratives.analysis import revision_data_layer as dl

OUT_DIR = notebook_dir / 'outputs' / 'revision'
FIG_DIR = OUT_DIR / 'figures'
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
RNG = np.random.default_rng(RNG_SEED)
print(f'Models in scope: {MODELS_TO_COMPARE}')
print(f'Anchor for stats:  {ANCHOR_MODEL}')


Models in scope: ['gpt-5', 'deepseek-r1', 'claude-sonnet-4-5-thinking']
Anchor for stats:  gpt-5


## 3. Load data

In [3]:
real = dl.load_real_from_schema()
print(f'Real ground truth: {len(real)} participants')

synth = {}
for tag in MODELS_TO_COMPARE:
    df = dl.load_synth_from_schema(tag)
    synth[tag] = df
    print(f'  {tag:<30} rows={len(df):>4}  runs={sorted(df["run_number"].unique().tolist())}')

def per_narrative_mean(df, col):
    return df.groupby('narrative_hash', as_index=False)[col].mean()

agg = {tag: {q: per_narrative_mean(df, q) for q in QUESTIONNAIRES}
       for tag, df in synth.items()}


Real ground truth: 152 participants


  gpt-5                          rows= 456  runs=[1, 2, 3]


  deepseek-r1                    rows= 456  runs=[1, 2, 3]


  claude-sonnet-4-5-thinking     rows= 456  runs=[1, 2, 3]


## 4. Helpers

In [4]:
def fisher_ci(r, n, alpha=0.05):
    if pd.isna(r) or n < 4:
        return (np.nan, np.nan)
    z = np.arctanh(r)
    se = 1.0 / np.sqrt(n - 3)
    crit = stats.norm.ppf(1 - alpha / 2)
    return (np.tanh(z - crit * se), np.tanh(z + crit * se))

def agreement(real_s, synth_s):
    m = ~(real_s.isna() | synth_s.isna())
    r = real_s[m].to_numpy(dtype=float)
    s = synth_s[m].to_numpy(dtype=float)
    n = len(r)
    if n < 3:
        return {'n': n, 'mae': np.nan, 'rmse': np.nan, 'pearson_r': np.nan,
                'r_lo': np.nan, 'r_hi': np.nan, 'spearman': np.nan}
    mae = float(np.mean(np.abs(r - s)))
    rmse = float(np.sqrt(np.mean((r - s) ** 2)))
    pr, _ = stats.pearsonr(r, s)
    sp, _ = stats.spearmanr(r, s)
    lo, hi = fisher_ci(pr, n)
    return {'n': n, 'mae': mae, 'rmse': rmse, 'pearson_r': pr,
            'r_lo': lo, 'r_hi': hi, 'spearman': sp}

def icc_2way_mixed(x, y):
    df = pd.DataFrame({'a': x, 'b': y}).dropna()
    if len(df) < 3:
        return np.nan
    n = len(df)
    subj_means = df.mean(axis=1)
    grand = df.stack().mean()
    bms = (((subj_means - grand) ** 2).sum() * 2) / (n - 1)
    jms = (((df.mean() - grand) ** 2).sum() * n)
    ems = (((df.sub(subj_means, axis=0)
             .sub(df.mean(axis=0), axis=1) + grand) ** 2).to_numpy().sum()) / (n - 1)
    return float((bms - ems) / (bms + ems + 2 * (jms - ems) / n))


## 5. Table 7 - per-model agreement with real

In [5]:
rows = []
for tag in MODELS_TO_COMPARE:
    for q in QUESTIONNAIRES:
        m = real[['narrative_hash', q]].merge(agg[tag][q], on='narrative_hash',
                                              suffixes=('_real', '_synth'))
        rows.append({'model': tag, 'questionnaire': q,
                     **agreement(m[f'{q}_real'], m[f'{q}_synth'])})
df_table7 = pd.DataFrame(rows)
df_table7.to_csv(OUT_DIR / '07_table7_per_model_agreement.csv', index=False)
print(df_table7.round(3).to_string(index=False))


                     model  questionnaire   n    mae   rmse  pearson_r  r_lo  r_hi  spearman
                     gpt-5      pcs_total 152 10.963 14.029      0.444 0.306 0.563     0.419
                     gpt-5 bpi_total_mean 152  1.083  1.398      0.699 0.608 0.773     0.709
                     gpt-5      tsk_total 152  6.654  8.354      0.344 0.195 0.477     0.324
               deepseek-r1      pcs_total 152 10.664 13.570      0.474 0.340 0.589     0.435
               deepseek-r1 bpi_total_mean 152  1.086  1.421      0.668 0.569 0.747     0.677
               deepseek-r1      tsk_total 152  9.121 11.223      0.312 0.161 0.449     0.281
claude-sonnet-4-5-thinking      pcs_total 152 10.752 13.744      0.470 0.336 0.586     0.450
claude-sonnet-4-5-thinking bpi_total_mean 152  0.967  1.241      0.726 0.641 0.794     0.734
claude-sonnet-4-5-thinking      tsk_total 152  7.114  8.785      0.368 0.221 0.498     0.321


## 6. Table 8 - pairwise inter-model agreement

In [6]:
pair_rows = []
for q in QUESTIONNAIRES:
    for a, b in combinations(MODELS_TO_COMPARE, 2):
        m = (agg[a][q].rename(columns={q: f'{q}_a'})
             .merge(agg[b][q].rename(columns={q: f'{q}_b'}), on='narrative_hash'))
        metrics = agreement(m[f'{q}_a'], m[f'{q}_b'])
        icc = icc_2way_mixed(m[f'{q}_a'], m[f'{q}_b'])
        pair_rows.append({'questionnaire': q, 'model_a': a, 'model_b': b,
                          **metrics, 'icc': icc})
df_table8 = pd.DataFrame(pair_rows)
df_table8.to_csv(OUT_DIR / '07_table8_inter_model_agreement.csv', index=False)
print(df_table8.round(3).to_string(index=False))


 questionnaire     model_a                    model_b   n   mae  rmse  pearson_r  r_lo  r_hi  spearman   icc
     pcs_total       gpt-5                deepseek-r1 152 5.004 7.229      0.898 0.863 0.925     0.872 0.847
     pcs_total       gpt-5 claude-sonnet-4-5-thinking 152 3.649 4.872      0.942 0.921 0.957     0.939 0.938
     pcs_total deepseek-r1 claude-sonnet-4-5-thinking 152 4.513 6.750      0.886 0.847 0.916     0.866 0.858
bpi_total_mean       gpt-5                deepseek-r1 152 0.929 1.087      0.953 0.935 0.965     0.951 0.815
bpi_total_mean       gpt-5 claude-sonnet-4-5-thinking 152 0.311 0.431      0.976 0.968 0.983     0.971 0.968
bpi_total_mean deepseek-r1 claude-sonnet-4-5-thinking 152 0.806 0.926      0.956 0.940 0.968     0.945 0.845
     tsk_total       gpt-5                deepseek-r1 152 5.213 6.166      0.877 0.834 0.909     0.860 0.696
     tsk_total       gpt-5 claude-sonnet-4-5-thinking 152 2.640 3.440      0.901 0.866 0.927     0.872 0.890
     tsk_total deep

## 7. Table 9 - statistical comparison vs anchor

In [7]:
def bootstrap_delta(real_v, anchor_v, other_v, iters, rng):
    delta = np.abs(real_v - anchor_v) - np.abs(real_v - other_v)
    idx = rng.integers(0, len(delta), size=(iters, len(delta)))
    means = delta[idx].mean(axis=1)
    return float(delta.mean()), tuple(np.quantile(means, [0.025, 0.975]))

stat_rows = []
for q in QUESTIONNAIRES:
    anc = agg[ANCHOR_MODEL][q].rename(columns={q: 'anchor'})
    for tag in MODELS_TO_COMPARE:
        if tag == ANCHOR_MODEL:
            continue
        oth = agg[tag][q].rename(columns={q: 'other'})
        m = (real[['narrative_hash', q]]
             .merge(anc, on='narrative_hash')
             .merge(oth, on='narrative_hash')
             .dropna(subset=[q, 'anchor', 'other']))
        r_v = m[q].astype(float).to_numpy()
        a_v = m['anchor'].astype(float).to_numpy()
        o_v = m['other'].astype(float).to_numpy()
        # zero_method='zsplit' is robust when many absolute-error pairs are tied
        # (common for BPI on the small Sonnet sample, which has a narrow score range).
        try:
            w_stat, w_p = stats.wilcoxon(np.abs(r_v - a_v), np.abs(r_v - o_v), zero_method='zsplit')
        except ValueError:
            w_stat, w_p = (np.nan, np.nan)
        mean_delta, (lo, hi) = bootstrap_delta(r_v, a_v, o_v, BOOTSTRAP_ITERS, RNG)
        stat_rows.append({'questionnaire': q, 'anchor': ANCHOR_MODEL, 'other': tag,
                          'n': len(m), 'wilcoxon_W': w_stat, 'wilcoxon_p': w_p,
                          'mean_delta_abs_err': mean_delta,
                          'bootstrap_95ci_lo': lo, 'bootstrap_95ci_hi': hi})
df_table9 = pd.DataFrame(stat_rows)
df_table9.to_csv(OUT_DIR / '07_table9_stat_comparison_vs_anchor.csv', index=False)
print(df_table9.round(4).to_string(index=False))


 questionnaire anchor                      other   n  wilcoxon_W  wilcoxon_p  mean_delta_abs_err  bootstrap_95ci_lo  bootstrap_95ci_hi
     pcs_total  gpt-5                deepseek-r1 152      5415.0      0.4629              0.2982            -0.6579             1.2697
     pcs_total  gpt-5 claude-sonnet-4-5-thinking 152      5559.5      0.6396              0.2105            -0.5329             0.9539
bpi_total_mean  gpt-5                deepseek-r1 152      5683.5      0.8103             -0.0027            -0.1501             0.1535
bpi_total_mean  gpt-5 claude-sonnet-4-5-thinking 152      3960.0      0.0006              0.1159             0.0527             0.1784
     tsk_total  gpt-5                deepseek-r1 152      2622.5      0.0000             -2.4671            -3.2237            -1.6491
     tsk_total  gpt-5 claude-sonnet-4-5-thinking 152      4819.0      0.0671             -0.4605            -0.9606             0.0219


## 8. Figures

In [8]:
# Forest plot of Pearson r.
fig, axes = plt.subplots(1, len(QUESTIONNAIRES), figsize=(4 * len(QUESTIONNAIRES), 4), sharey=True)
for ax, q in zip(np.atleast_1d(axes), QUESTIONNAIRES):
    sub = df_table7.query('questionnaire == @q').reset_index(drop=True)
    y_pos = np.arange(len(sub))
    ax.errorbar(sub['pearson_r'], y_pos,
                xerr=[sub['pearson_r'] - sub['r_lo'], sub['r_hi'] - sub['pearson_r']],
                fmt='o', capsize=4)
    ax.set_yticks(y_pos); ax.set_yticklabels(sub['model'])
    ax.axvline(0, color='grey', linewidth=0.5)
    ax.set_title(q); ax.set_xlabel('Pearson r (95% CI)')
fig.tight_layout(); fig.savefig(FIG_DIR / '07_fig1_forest_plot.pdf', bbox_inches='tight'); plt.close(fig)
print('Saved forest plot.')


Saved forest plot.


In [9]:
# Bland-Altman grid.
fig, axes = plt.subplots(len(QUESTIONNAIRES), len(MODELS_TO_COMPARE),
                         figsize=(4 * len(MODELS_TO_COMPARE), 3.5 * len(QUESTIONNAIRES)),
                         squeeze=False)
for i, q in enumerate(QUESTIONNAIRES):
    for j, tag in enumerate(MODELS_TO_COMPARE):
        m = real[['narrative_hash', q]].merge(agg[tag][q], on='narrative_hash',
                                              suffixes=('_real', '_synth'))
        mean_v = (m[f'{q}_real'] + m[f'{q}_synth']) / 2
        diff_v = m[f'{q}_synth'] - m[f'{q}_real']
        mu, sd = diff_v.mean(), diff_v.std()
        ax = axes[i][j]
        ax.scatter(mean_v, diff_v, alpha=0.5, s=15)
        ax.axhline(mu, color='black', linewidth=0.8)
        ax.axhline(mu + 1.96 * sd, color='grey', linewidth=0.6, linestyle='--')
        ax.axhline(mu - 1.96 * sd, color='grey', linewidth=0.6, linestyle='--')
        ax.set_title(f'{tag} - {q}', fontsize=9)
        ax.set_xlabel('mean'); ax.set_ylabel('synth - real')
fig.tight_layout(); fig.savefig(FIG_DIR / '07_fig2_bland_altman.pdf', bbox_inches='tight'); plt.close(fig)
print('Saved Bland-Altman grid.')


Saved Bland-Altman grid.


In [10]:
# Error boxplots.
fig, axes = plt.subplots(1, len(QUESTIONNAIRES), figsize=(4 * len(QUESTIONNAIRES), 4))
for ax, q in zip(np.atleast_1d(axes), QUESTIONNAIRES):
    boxes, labels = [], []
    for tag in MODELS_TO_COMPARE:
        m = real[['narrative_hash', q]].merge(agg[tag][q], on='narrative_hash',
                                              suffixes=('_real', '_synth'))
        boxes.append((m[f'{q}_synth'] - m[f'{q}_real']).dropna().to_numpy())
        labels.append(tag)
    ax.boxplot(boxes, labels=labels, showmeans=True)
    ax.axhline(0, color='grey', linewidth=0.5)
    ax.set_title(q); ax.set_ylabel('synth - real')
    ax.tick_params(axis='x', rotation=20)
fig.tight_layout(); fig.savefig(FIG_DIR / '07_fig3_error_boxplots.pdf', bbox_inches='tight'); plt.close(fig)
print('Saved error boxplots.')


Saved error boxplots.


## 9. Done

In [11]:
print('Tables written:')
for name in ('07_table7_per_model_agreement.csv',
             '07_table8_inter_model_agreement.csv',
             '07_table9_stat_comparison_vs_anchor.csv'):
    print(f'  - {OUT_DIR / name}')
print('Figures written under', FIG_DIR)


Tables written:
  - /Users/gferreira/Documents/my_repos/pain-narratives-app-public/notebooks/outputs/revision/07_table7_per_model_agreement.csv
  - /Users/gferreira/Documents/my_repos/pain-narratives-app-public/notebooks/outputs/revision/07_table8_inter_model_agreement.csv
  - /Users/gferreira/Documents/my_repos/pain-narratives-app-public/notebooks/outputs/revision/07_table9_stat_comparison_vs_anchor.csv
Figures written under /Users/gferreira/Documents/my_repos/pain-narratives-app-public/notebooks/outputs/revision/figures
